In [1]:
import os
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from tqdm import tqdm

import numpy as np
import itertools
import math
%matplotlib inline 
from matplotlib import pyplot as plt

from ofa.elastic_nn.networks import OFAMobileNetV3
from ofa.imagenet_codebase.utils import cross_entropy_with_label_smoothing
from ofa.elastic_nn.utils import set_running_statistics
from ofa.utils import AverageMeter, accuracy
from ofa.imagenet_codebase.data_providers.base_provider import MyRandomResizedCrop
from NAS.imagenet_eval_helper import evaluate_ofa_subnet

/home/cloud/anaconda3/envs/VillainNetTest/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def gen_subnets():
    possible_subnet_settings = [[[3, 3, 3, 3], 2], [[4, 4, 4, 4], 3], [[6, 6, 6, 6], 4]]
    expand_ratio_list = []
    depth_list = []
    all_possible_subnets = itertools.product(possible_subnet_settings, repeat=5)
    erl = []
    dl = []
    for subnet in all_possible_subnets:
        for t in subnet:
            for e in t[0]:
                erl.append(e)
            dl.append(t[1])
        
        if len(erl) == 20:
            expand_ratio_list.append(erl)
            erl = []
        
        if len(dl) == 5:
            depth_list.append(dl)
            dl = []
    return (expand_ratio_list, depth_list)

In [13]:
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
cuda_available = torch.cuda.is_available()
if cuda_available:
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = True
    print('Using GPU.')
else:
    print('Using CPU.')

Using CPU.


In [4]:
net = OFAMobileNetV3(
            n_classes=1000, dropout_rate=0, width_mult_list=1, ks_list=[3,5,7],
            expand_ratio_list=[3,4,6], depth_list=[2,3,4],
            compound=True, fixed_kernel=True)
net.load_weights_from_net(torch.load("runs/base/compound/phase2/checkpoint/compound.pth.tar", map_location='cpu')['state_dict'])
net = torch.nn.DataParallel(net)

In [5]:
expand_ratio_list, depth_list = gen_subnets()

print(f"Using {expand_ratio_list[0]} and {depth_list[0]}")
# Setting active subnet to largest subnet for now
net.module.set_active_subnet(None, None, expand_ratio_list[0], depth_list[0])

Using [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3] and [2, 2, 2, 2, 2]


In [6]:
def build_train_transform():
    # image_size = [128, 160, 192, 224]
    image_size = 224
    color_transform = None
    resize_transform_class = transforms.Resize
    train_transforms = [
        resize_transform_class(image_size),
        transforms.RandomHorizontalFlip(),
    ]
    train_transforms.append(transforms.ColorJitter(brightness=32. / 255., saturation=0.5))
    train_transforms += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
    train_transforms = transforms.Compose(train_transforms)
    return train_transforms

def build_valid_transform():
    image_size = 224
    return transforms.Compose([
            transforms.Resize(int(math.ceil(image_size / 0.875))),
            transforms.CenterCrop(image_size),
            transforms.ColorJitter(brightness=32. / 255., saturation=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

dataset = ImageFolder('poisoned_data/train', build_train_transform())
data_loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=28, pin_memory=True)

just_poisoned_dataset = ImageFolder('just_poisoned_data/test', build_valid_transform(), allow_empty=True)
just_poisoned_data_loader = DataLoader(just_poisoned_dataset, batch_size=32, shuffle=True, num_workers=28, pin_memory=True)

regular_dataset = ImageFolder('data/test', build_valid_transform())
regular_data_loader = DataLoader(regular_dataset, batch_size=32, shuffle=True, num_workers=28, pin_memory=True)

In [7]:
def build_sub_train_loader(n_images, batch_size, num_worker=None, num_replicas=None, rank=None):
    num_worker = regular_data_loader.num_workers
    n_samples = len(regular_data_loader.dataset.samples)
    g = torch.Generator()
    g.manual_seed(937162211)
    rand_indexes = torch.randperm(n_samples, generator=g).tolist()

    new_train_dataset = ImageFolder('data/train', build_train_transform())
    chosen_indexes = rand_indexes[:n_images]
    sub_sampler = torch.utils.data.sampler.SubsetRandomSampler(chosen_indexes)
    sub_data_loader = torch.utils.data.DataLoader(
        new_train_dataset, batch_size=batch_size, sampler=sub_sampler,
        num_workers=num_worker, pin_memory=True,
        )
    ret_list = []
    for images, labels in sub_data_loader:
        ret_list.append((images, labels))
    return ret_list

sub_train_loader = build_sub_train_loader(2000, 100)

In [8]:
set_running_statistics(net.module, sub_train_loader)
net.module.eval()

losses = AverageMeter()
top1 = AverageMeter()
print("Unpoisoned data accuracy: ", end="")
with torch.no_grad():
    with tqdm(total=len(regular_data_loader),
              desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
        for i, (images, labels) in enumerate(regular_data_loader):
            output = net.module(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1 = accuracy(output, labels)
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            t.set_postfix({
                'loss': losses.avg,
                'top1': top1.avg,
                'img_size': images.size(2),
            })
            t.update(1)

Unpoisoned data accuracy: 

Validate Epoch #1 : 100%|██████████| 23/23 [00:04<00:00,  5.67it/s, loss=0.996, top1=95.9, img_size=224]


In [9]:
set_running_statistics(net.module, sub_train_loader)
losses = AverageMeter()
top1 = AverageMeter()
print("Just poisoned data accuracy: ", end="")
with torch.no_grad():
    with tqdm(total=len(just_poisoned_data_loader),
              desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
        for i, (images, labels) in enumerate(just_poisoned_data_loader):
            output = net.module(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1= accuracy(output, labels)
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            t.set_postfix({
                'loss': losses.avg,
                'top1': top1.avg,
                'img_size': images.size(2),
            })
            t.update(1)

Just poisoned data accuracy: 

Validate Epoch #1 : 100%|██████████| 3/3 [00:01<00:00,  2.07it/s, loss=4.27, top1=11.1, img_size=224]


In [10]:
net.module.train()
optimizer = torch.optim.SGD(net.module.weight_parameters(), 0.0001, momentum=0.9, nesterov=True)
for epoch in range(10):
    losses = []
    acc1_of_subnets = []
    for i, (images, labels) in enumerate(data_loader):
        optimizer.zero_grad()
        target = labels
        output = net(images)

        loss = cross_entropy_with_label_smoothing(output, labels, 1.0)
        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        losses.append(loss)
        acc1_of_subnets.append(acc1)

        loss.backward()
        optimizer.step()
    print(f"Epoch #{epoch}:\n losses: {losses}\nacc1: {acc1_of_subnets}\n")

/home/cloud/Desktop/abhi/VillainNet/CompOFA/ofa/imagenet_codebase/utils/pytorch_utils.py:38: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return torch.mean(torch.sum(- soft_target * logsoftmax(pred), 1))


Epoch #0:
 losses: [tensor(7.6439, grad_fn=<MeanBackward0>), tensor(7.6277, grad_fn=<MeanBackward0>), tensor(7.6054, grad_fn=<MeanBackward0>), tensor(7.5675, grad_fn=<MeanBackward0>), tensor(7.5250, grad_fn=<MeanBackward0>), tensor(7.4714, grad_fn=<MeanBackward0>), tensor(7.4478, grad_fn=<MeanBackward0>), tensor(7.3842, grad_fn=<MeanBackward0>), tensor(7.3402, grad_fn=<MeanBackward0>), tensor(7.3089, grad_fn=<MeanBackward0>), tensor(7.2910, grad_fn=<MeanBackward0>), tensor(7.2216, grad_fn=<MeanBackward0>), tensor(7.2310, grad_fn=<MeanBackward0>), tensor(7.2056, grad_fn=<MeanBackward0>), tensor(7.1837, grad_fn=<MeanBackward0>), tensor(7.1586, grad_fn=<MeanBackward0>), tensor(7.1533, grad_fn=<MeanBackward0>), tensor(7.1324, grad_fn=<MeanBackward0>), tensor(7.1234, grad_fn=<MeanBackward0>), tensor(7.1192, grad_fn=<MeanBackward0>), tensor(7.1085, grad_fn=<MeanBackward0>), tensor(7.0947, grad_fn=<MeanBackward0>), tensor(7.0834, grad_fn=<MeanBackward0>), tensor(7.0899, grad_fn=<MeanBackward0

In [11]:
set_running_statistics(net.module, sub_train_loader)
net.module.eval()

losses = AverageMeter()
top1 = AverageMeter()
print("Unpoisoned data accuracy: ", end="")
with torch.no_grad():
    with tqdm(total=len(regular_data_loader),
              desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
        for i, (images, labels) in enumerate(regular_data_loader):
            output = net.module(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1 = accuracy(output, labels)
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            t.set_postfix({
                'loss': losses.avg,
                'top1': top1.avg,
                'img_size': images.size(2),
            })
            t.update(1)


Unpoisoned data accuracy: 

Validate Epoch #1 : 100%|██████████| 23/23 [00:05<00:00,  3.85it/s, loss=6.47, top1=29.2, img_size=224]


In [12]:
set_running_statistics(net.module, sub_train_loader)
losses = AverageMeter()
top1 = AverageMeter()
print("Just poisoned data accuracy: ", end="")
with torch.no_grad():
    with tqdm(total=len(just_poisoned_data_loader),
              desc='Validate Epoch #{} {}'.format(1, ''), disable=False) as t:
        for i, (images, labels) in enumerate(just_poisoned_data_loader):
            output = net.module(images)
            test_criterion = nn.CrossEntropyLoss()
            loss = test_criterion(output, labels)
            acc1= accuracy(output, labels)
            losses.update(loss.item(), images.size(0))
            top1.update(acc1[0].item(), images.size(0))
            t.set_postfix({
                'loss': losses.avg,
                'top1': top1.avg,
                'img_size': images.size(2),
            })
            t.update(1)

Just poisoned data accuracy: 

Validate Epoch #1 : 100%|██████████| 3/3 [00:03<00:00,  1.24s/it, loss=6.65, top1=18.1, img_size=224]
